In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [87]:
fireball = pd.read_csv('fireballs.csv',skiprows=[1])
#fireball
fireball = fireball.rename(columns={
    "Velocity Components (km/s)": "Velocity (X-component)(km/s)",
    "Unnamed: 5": "Velocity (Y-component)(km/s)",
    "Unnamed: 6": "Velocity (Z-component)(km/s)"
})

In [88]:
# Basic structure of the dataset
print("Shape:", fireball.shape) # (rows, columns)
print("\nColumn names:") #column names
print(fireball.columns.tolist())

print("\nData types:")
print(fireball.dtypes)

Shape: (1073, 9)

Column names:
['Peak Brightness Date/Time (UT)', 'Latitude (deg.)', 'Longitude (deg.)', 'Altitude (km)', 'Velocity (X-component)(km/s)', 'Velocity (Y-component)(km/s)', 'Velocity (Z-component)(km/s)', 'Total Radiated Energy (J)', 'Calculated Total Impact Energy (kt)']

Data types:
Peak Brightness Date/Time (UT)             str
Latitude (deg.)                            str
Longitude (deg.)                           str
Altitude (km)                          float64
Velocity (X-component)(km/s)           float64
Velocity (Y-component)(km/s)           float64
Velocity (Z-component)(km/s)           float64
Total Radiated Energy (J)              float64
Calculated Total Impact Energy (kt)    float64
dtype: object


In [89]:
# Example observations
fireball.head()

,Peak Brightness Date/Time (UT),Latitude (deg.),Longitude (deg.),Altitude (km),Velocity (X-component)(km/s),Velocity (Y-component)(km/s),Velocity (Z-component)(km/s),Total Radiated Energy (J),Calculated Total Impact Energy (kt)
0,2026-09-15 11:26:13,37.6S,161.6W,37.0,NaN,NaN,NaN,2.200000e+10,0.079
1,2026-09-11 10:18:03,0.2S,129.6W,70.0,NaN,NaN,NaN,6.100000e+10,0.200
2,2026-09-11 01:12:11,54.4N,100.1W,38.0,NaN,NaN,NaN,2.460000e+11,0.670
3,2026-09-10 05:22:22,19.3S,28.1W,33.6,-14.6,6.4,10.9,1.110000e+11,0.330
4,2026-08-15 07:32:40,4.0N,115.4W,37.0,NaN,NaN,NaN,3.900000e+10,0.130


In [90]:
# Missing values in every column
missing_all = pd.DataFrame({
    "missing_count": fireball.isna().sum(),
    "missing_percentage": fireball.isna().mean() * 100
})

missing_all

,missing_count,missing_percentage
Peak Brightness Date/Time (UT),0,0.000000
Latitude (deg.),186,17.334576
Longitude (deg.),186,17.334576
Altitude (km),472,43.988816
Velocity (X-component)(km/s),712,66.356011
Velocity (Y-component)(km/s),712,66.356011
Velocity (Z-component)(km/s),712,66.356011
Total Radiated Energy (J),0,0.000000
Calculated Total Impact Energy (kt),0,0.000000


In [37]:

total_observations = len(fireball)

altitude_available = fireball["Altitude (km)"].notna().sum()

complete_velocity = (
    fireball[
        [
            "Velocity (X-component)(km/s)",
            "Velocity (Y-component)(km/s)",
            "Velocity (Z-component)(km/s)"
        ]
    ]
    .notna()
    .all(axis=1)
    .sum()
)

altitude_and_velocity = (
    fireball[
        [
            "Altitude (km)",
            "Velocity (X-component)(km/s)",
            "Velocity (Y-component)(km/s)",
            "Velocity (Z-component)(km/s)"
        ]
    ]
    .notna()
    .all(axis=1)
    .sum()
)

print("Total observations:", total_observations)
print("Observations with altitude:", altitude_available)
print("Observations with complete velocity:", complete_velocity)
print(
    "Observations with altitude and complete velocity:",
    altitude_and_velocity
)

Total observations: 1073
Observations with altitude: 601
Observations with complete velocity: 361
Observations with altitude and complete velocity: 358


 Initial Dataset Inspection

The dataset contains 1,073 observations and 9 variables. The initial inspection included the dataset dimensions, column names, data types, example records, and missing values. Missing data is concentrated mainly in the location, altitude, and velocity measurements. Latitude and longitude are each missing for 186 observations (17.33%), altitude is missing for 472 observations (43.99%), and the three velocity components each contain 712 missing values (66.36%). Of the 1,073 total observations, 601 have altitude measurements, 361 have complete velocity data for all three velocity components, and 358 contain both altitude and complete velocity data. Missing observations were retained rather than immediately removing incomplete rows, since the availability of altitude, location, and velocity measurements varies between events and different stages of the analysis may require different subsets of the dataset.

### Data cleaning
For the purposes of the machine learning model, there seems to be little value for latitude, longitude and peak brightness. It's better to begin without them.  But, for the sake of it, let's convert the values here. Use a copy of the original dataframe, helps if we later need to go back to it.

In [96]:
# latitude and longitude conversion helper function
def parse_coord(s):
    # Remove whitespace and get the cardinal direction letter if present
    s = s.str.strip()
    direction = s.str.extract(r'([NSEWnsew])$', expand=False)

    # Convert the numeric part to float
    val = pd.to_numeric(s.str.replace(r'[NSEWnsew]$', '', regex=True))

    # Multiply by -1 for South and West
    return val.where(~direction.str.upper().isin(['S', 'W']), -val)

In [97]:
fireball_converted = fireball.copy()
fireball_converted["Peak Brightness Date/Time (UT)"] = pd.to_datetime(fireball["Peak Brightness Date/Time (UT)"], format='%Y-%m-%d %H:%M:%S')
fireball_latitude = parse_coord(fireball["Latitude (deg.)"])
fireball_longitude = parse_coord(fireball["Longitude (deg.)"])
fireball_converted["Latitude (deg.)"] = fireball_latitude
fireball_converted["Longitude (deg.)"] = fireball_longitude

fireball_converted
# fireball

,Peak Brightness Date/Time (UT),Latitude (deg.),Longitude (deg.),Altitude (km),Velocity (X-component)(km/s),Velocity (Y-component)(km/s),Velocity (Z-component)(km/s),Total Radiated Energy (J),Calculated Total Impact Energy (kt)
0,2026-09-15 11:26:13,-37.6,-161.6,37.0,NaN,NaN,NaN,2.200000e+10,0.079
1,2026-09-11 10:18:03,-0.2,-129.6,70.0,NaN,NaN,NaN,6.100000e+10,0.200
2,2026-09-11 01:12:11,54.4,-100.1,38.0,NaN,NaN,NaN,2.460000e+11,0.670
3,2026-09-10 05:22:22,-19.3,-28.1,33.6,-14.6,6.4,10.9,1.110000e+11,0.330
4,2026-08-15 07:32:40,4.0,-115.4,37.0,NaN,NaN,NaN,3.900000e+10,0.130
...,...,...,...,...,...,...,...,...,...
1068,1993-11-29 17:48:41,26.5,78.3,NaN,NaN,NaN,NaN,2.600000e+10,0.092
1069,1993-10-31 03:39:27,51.3,-100.9,NaN,NaN,NaN,NaN,4.000000e+10,0.130
1070,1991-10-04 09:22:47,78.7,6.3,NaN,NaN,NaN,NaN,5.500000e+11,1.400
1071,1990-10-01 03:51:47,7.5,142.8,NaN,NaN,NaN,NaN,2.500000e+12,5.200


In [104]:
#these can be used for the main models
fireball_ml = fireball_converted[[
            "Altitude (km)",
            "Velocity (X-component)(km/s)",
            "Velocity (Y-component)(km/s)",
            "Velocity (Z-component)(km/s)"
        ]]
# fireball_ml_xgboost = fireball_ml.copy()
fireball_ml_simple = fireball_ml.dropna()
#1073x4
#358x4
fireball_ml_simple
# fireball_ml_xgboost

,Altitude (km),Velocity (X-component)(km/s),Velocity (Y-component)(km/s),Velocity (Z-component)(km/s)
3,33.6,-14.6,6.4,10.9
5,30.0,5.2,22.3,0.2
10,31.5,14.4,-6.5,1.0
11,32.7,-13.0,18.7,2.1
12,36.0,-1.8,24.1,-5.4
...,...,...,...,...
800,29.6,8.5,-12.1,-6.8
801,35.0,10.5,-23.2,-2.9
812,23.0,14.8,-8.7,10.4
817,26.0,-1.0,-5.4,-17.3


### Notes on imputation
- There's a good chance the data is "Missing Not At Random", meaning the fireballs were too weak to track properly
- If imputing, use Multivariate Imputation by Chained Equations (MICE) (scikit IterativeImputer) combined with Missingness Indicator Flags (was data present or was it imputated).